# GroupDNA – WhatsApp Group Chat Analyzer

Name: Shiven   
Batch: november  
Date: 21 November 2026
## Project Description
GroupDNA is a WhatsApp group chat analyzer built using Python fundamentals and NumPy. It analyzes messaging patterns, group activity, frequently used words, response times, silent streaks and personality archetypes.

In [2]:
# GroupDNA - WhatsApp Group Chat Analyzer
import numpy as np
from datetime import datetime, timedelta

# FEATURE 1: CHAT PARSER

def is_date_line(line):
    if len(line) < 17:
        return False

    first = line[:8]

    return (
        first[0:2].isdigit() and first[2] == "/" and
        first[3:5].isdigit() and first[5] == "/" and
        first[6:8].isdigit()
    )

def parse_chat(filename):

    messages = []
    system_count = 0
    deleted_count = {}
    media_count = {}

    with open(filename, "r", encoding="utf-8") as file:
        lines = file.readlines()

    previous_message = None

    for line in lines:

        line = line.rstrip("\n").rstrip("\r")

        if line.strip() == "":
            continue

        # Handle multiline messages
        if not is_date_line(line):
            if previous_message is not None:
                previous_message["text"] += " " + line.strip()
            continue

        parts = line.split(" - ", 1)

        if len(parts) != 2:
            system_count += 1
            previous_message = None
            continue

        timestamp = parts[0]
        remaining = parts[1]

        sender_parts = remaining.split(": ", 1)

        if len(sender_parts) != 2:
            system_count += 1
            previous_message = None
            continue

        sender = sender_parts[0].strip()
        message_text = sender_parts[1].strip()

        try:
            datetime.strptime(timestamp, "%d/%m/%y, %H:%M")
        except ValueError:
            system_count += 1
            previous_message = None
            continue

        # Deleted messages are counted separately
        if message_text == "This message was deleted":
            deleted_count[sender] = deleted_count.get(sender, 0) + 1
            previous_message = None
            continue

        # Media messages are counted but excluded from text analysis
        is_media = message_text == "<Media omitted>"

        if is_media:
            media_count[sender] = media_count.get(sender, 0) + 1

        message = {
            "timestamp": timestamp,
            "sender": sender,
            "text": message_text,
            "datetime": datetime.strptime(
                timestamp, "%d/%m/%y, %H:%M"
            ),
            "media": is_media
        }

        messages.append(message)
        previous_message = message

    return messages, system_count, deleted_count, media_count


# Upload dataset in Google Colab
from google.colab import files

print("Please upload hostel_bois.txt")
uploaded = files.upload()

if not uploaded:
    raise ValueError("Please upload hostel_bois.txt to continue.")

filename = next(iter(uploaded))

messages, system_count, deleted_count, media_count = parse_chat(filename)

if len(messages) == 0:
    raise ValueError("No valid messages found in the uploaded file.")

participants = sorted(set(m["sender"] for m in messages))

messages.sort(key=lambda x: x["datetime"])

print("\nPARSER RESULTS")
print("=" * 60)
print("Successfully parsed:", len(messages), "messages")
print("Participants:", len(participants))
print("System messages skipped:", system_count)
print("Deleted messages:", sum(deleted_count.values()))
print("Media messages:", sum(media_count.values()))

print("\nFIRST 5 MESSAGES")
for message in messages[:5]:
    print(message["timestamp"], message["sender"], message["text"])

print("\nLAST 5 MESSAGES")
for message in messages[-5:]:
    print(message["timestamp"], message["sender"], message["text"])

# FEATURE 2: GROUP OVERVIEW
message_counts = {}
person_messages = {}

for person in participants:
    message_counts[person] = 0
    person_messages[person] = []

for message in messages:
    person = message["sender"]
    message_counts[person] += 1
    person_messages[person].append(message)

total_messages = len(messages)
first_date = messages[0]["datetime"].date()
last_date = messages[-1]["datetime"].date()

total_days = (last_date - first_date).days + 1

print("\n" + "=" * 65)
print("                 GROUP OVERVIEW")
print("=" * 65)

print("Group              : Hostel Bois 4ever")
print("Period             :", first_date.strftime("%d %B %Y"),
      "to", last_date.strftime("%d %B %Y"))
print("Total days         :", total_days)
print("Total messages     :", total_messages)
print("Participants       :", len(participants))

print("\nMESSAGES PER PERSON")

sorted_people = sorted(
    message_counts,
    key=lambda person: message_counts[person],
    reverse=True
)

for person in sorted_people:
    percentage = message_counts[person] / total_messages * 100
    bar = "█" * int(20 * message_counts[person] / max(message_counts.values()))

    print(f"{person:<10} {bar:<22} "
          f"{message_counts[person]:>5} ({percentage:.1f}%)")

# FEATURE 3: MOST ACTIVE DAY AND HOUR

day_counts = {}
hour_counts = {}

for message in messages:

    day = message["datetime"].date()
    hour = message["datetime"].hour

    day_counts[day] = day_counts.get(day, 0) + 1
    hour_counts[hour] = hour_counts.get(hour, 0) + 1

busiest_day = max(day_counts, key=day_counts.get)
busiest_hour = max(hour_counts, key=hour_counts.get)

print("\n" + "=" * 65)
print("              MOST ACTIVE DAY AND HOUR")
print("=" * 65)

print("Busiest day :",
      busiest_day.strftime("%d %B %Y"),
      f"({day_counts[busiest_day]} messages)")

print("Busiest hour:",
      f"{busiest_hour:02d}:00 - {(busiest_hour + 1) % 24:02d}:00",
      f"({hour_counts[busiest_hour]} messages)")

# FEATURE 4: ACTIVITY HEATMAP USING NUMPY

heatmap = np.zeros((len(participants), 24), dtype=int)

person_index = {}

for i, person in enumerate(participants):
    person_index[person] = i

for message in messages:

    person = message["sender"]
    hour = message["datetime"].hour

    heatmap[person_index[person]][hour] += 1

print("\n" + "=" * 75)
print("                  ACTIVITY HEATMAP")
print("=" * 75)

print("Person    ", end="")

for hour in range(24):
    print(f"{hour:02d} ", end="")

print()

for i, person in enumerate(participants):

    maximum = np.max(heatmap[i])

    print(f"{person:<10}", end="")

    for hour in range(24):

        value = heatmap[i][hour]

        if maximum == 0 or value == 0:
            symbol = "."
        elif value <= maximum * 0.25:
            symbol = "░"
        elif value <= maximum * 0.50:
            symbol = "▒"
        else:
            symbol = "█"

        print(f"{symbol:>2} ", end="")

    print()

print("\nTotal messages per person by NumPy:")
for i, person in enumerate(participants):
    print(person, ":", np.sum(heatmap[i]))


# ------------------------------------------------------------
# FEATURE 5: TOP WORDS
# ------------------------------------------------------------

stop_words = [
    "i", "is", "the", "a", "and", "or", "to", "of",
    "in", "on", "for", "it", "this", "that", "are",
    "was", "am", "you", "my", "me", "we", "he", "she",
    "they", "be", "have", "has", "had", "with", "at",
    "but", "so", "as", "an", "do", "not", "what"
]

punctuation = ".,!?;:'\"()[]{}<>-_/\\@#$%^&*+=~`|"

word_counts = {}
personal_word_counts = {}

for person in participants:
    personal_word_counts[person] = {}

for message in messages:

    if message["media"]:
        continue

    person = message["sender"]

    words = message["text"].lower().split()

    for word in words:

        word = word.strip(punctuation)

        if word == "" or word in stop_words:
            continue

        word_counts[word] = word_counts.get(word, 0) + 1

        personal_word_counts[person][word] = (
            personal_word_counts[person].get(word, 0) + 1
        )

top_words = sorted(
    word_counts.items(),
    key=lambda item: item[1],
    reverse=True
)[:10]

print("\n" + "=" * 65)
print("               THIS GROUP'S FAVOURITE WORDS")
print("=" * 65)

if top_words:
    maximum_word_count = top_words[0][1]

    for word, count in top_words:

        bar = "█" * int(25 * count / maximum_word_count)

        print(f"{word:<15} {bar:<27} {count}")
else:
    print("No words available for analysis.")

print("\nTOP 5 WORDS PER PERSON")

for person in participants:

    top_person_words = sorted(
        personal_word_counts[person].items(),
        key=lambda item: item[1],
        reverse=True
    )[:5]

    print("\n", person, ":")

    for word, count in top_person_words:
        print(f"  {word:<15} {count}")


# ------------------------------------------------------------
# FEATURE 6: RESPONSE SPEED AND SILENT STREAKS
# ------------------------------------------------------------

response_times = {}

for person in participants:
    response_times[person] = []

for i in range(1, len(messages)):

    previous = messages[i - 1]
    current = messages[i]

    if previous["sender"] != current["sender"]:

        person = current["sender"]

        gap = (
            current["datetime"] - previous["datetime"]
        ).total_seconds()

        if gap >= 0:
            response_times[person].append(gap)

average_response = {}

for person in participants:

    if response_times[person]:
        average_response[person] = (
            sum(response_times[person]) /
            len(response_times[person])
        )
    else:
        average_response[person] = None

valid_responses = {
    person: time
    for person, time in average_response.items()
    if time is not None
}

print("\n" + "=" * 65)
print("                 RESPONSE PATTERNS")
print("=" * 65)

if valid_responses:

    fastest = min(valid_responses, key=valid_responses.get)
    slowest = max(valid_responses, key=valid_responses.get)

    print("Fastest replier:", fastest,
          f"({valid_responses[fastest] / 60:.2f} minutes)")

    print("Slowest replier:", slowest,
          f"({valid_responses[slowest] / 3600:.2f} hours)")

else:
    print("Not enough messages to calculate response times.")


# Calculate active days and silent streaks

active_days = {}

for person in participants:
    active_days[person] = set()

for message in messages:

    person = message["sender"]
    day = message["datetime"].date()

    active_days[person].add(day)

silent_streaks = {}
silent_streak_dates = {}

for person in participants:

    longest_streak = 0
    current_streak = 0
    streak_start = None
    best_start = None
    best_end = None

    for day_number in range(total_days):

        current_date = first_date + timedelta(days=day_number)

        if current_date not in active_days[person]:

            if current_streak == 0:
                streak_start = current_date

            current_streak += 1

            if current_streak > longest_streak:
                longest_streak = current_streak
                best_start = streak_start
                best_end = current_date

        else:
            current_streak = 0

    silent_streaks[person] = longest_streak
    silent_streak_dates[person] = (best_start, best_end)

print("\nLONGEST SILENT STREAKS")

for person in sorted(
    participants,
    key=lambda p: silent_streaks[p],
    reverse=True
):

    start, end = silent_streak_dates[person]

    if silent_streaks[person] > 0:
        print(f"{person:<10}: {silent_streaks[person]} days "
              f"({start.strftime('%d %b')} - {end.strftime('%d %b')})")
    else:
        print(f"{person:<10}: 0 days (never went silent)")


# ------------------------------------------------------------
# FEATURE 7: PERSONALITY ARCHETYPE DETECTION
# ------------------------------------------------------------

def calculate_burst(person):

    bursts = []
    current_burst = 0

    for message in messages:

        if message["sender"] == person:
            current_burst += 1

        else:
            if current_burst > 0:
                bursts.append(current_burst)

            current_burst = 0

    if current_burst > 0:
        bursts.append(current_burst)

    if len(bursts) == 0:
        return 0

    return sum(bursts) / len(bursts)


def caring_score(person):

    keywords = [
        "okay", "safe", "eat", "sleep", "take care",
        "are you", "please", "reminder",
        "drink water", "don't forget"
    ]

    count = 0

    for message in person_messages[person]:

        if message["media"]:
            continue

        text = message["text"].lower()

        for keyword in keywords:
            if keyword in text:
                count += 1

    return count


def night_owl_score(person):

    night_messages = 0

    for message in person_messages[person]:

        hour = message["datetime"].hour

        if hour >= 23 or hour <= 4:
            night_messages += 1

    if message_counts[person] == 0:
        return 0

    return night_messages / message_counts[person]


def storyteller_score(person):

    total_words = 0
    valid_count = 0

    for message in person_messages[person]:

        if not message["media"]:
            total_words += len(message["text"].split())
            valid_count += 1

    if valid_count == 0:
        return 0

    return total_words / valid_count


def drama_score(person):

    drama_count = 0

    for message in person_messages[person]:

        if message["media"]:
            continue

        text = message["text"]

        alphabetic = ""

        for character in text:
            if character.isalpha():
                alphabetic += character

        all_caps = (
            len(alphabetic) >= 3 and alphabetic.isupper()
        )

        many_exclamations = text.count("!") >= 2

        if all_caps or many_exclamations:
            drama_count += 1

    if message_counts[person] == 0:
        return 0

    return drama_count / message_counts[person]


def ghost_score(person):

    silent_days = total_days - len(active_days[person])

    return silent_days / total_days


def comedian_score(person):

    funny_words = ["lol", "lmao", "haha", "rofl", "lmfao"]

    count = 0

    for message in person_messages[person]:

        if message["media"]:
            continue

        text = message["text"].lower()

        for word in text.split():
            word = word.strip(punctuation)

            if word in funny_words:
                count += 1

    if message_counts[person] == 0:
        return 0

    return count / message_counts[person]


def question_score(person):

    count = 0

    for message in person_messages[person]:

        if not message["media"] and message["text"].endswith("?"):
            count += 1

    if message_counts[person] == 0:
        return 0

    return count / message_counts[person]


# Calculate scores for every participant

archetype_scores = {}

for person in participants:

    archetype_scores[person] = {
        "THE SPAMMER": calculate_burst(person),
        "THE GROUP MOM": caring_score(person),
        "THE NIGHT OWL": night_owl_score(person),
        "THE STORYTELLER": storyteller_score(person),
        "THE DRAMA QUEEN": drama_score(person),
        "THE GHOST": ghost_score(person),
        "THE COMEDIAN": comedian_score(person),
        "THE QUESTION MASTER": question_score(person)
    }


# Normalize each archetype's scores across participants.
# This allows different kinds of measurements to be compared.

archetypes = [
    "THE SPAMMER",
    "THE GROUP MOM",
    "THE NIGHT OWL",
    "THE STORYTELLER",
    "THE DRAMA QUEEN",
    "THE GHOST",
    "THE COMEDIAN",
    "THE QUESTION MASTER"
]

normalized_scores = {}

for person in participants:
    normalized_scores[person] = {}

for archetype in archetypes:

    maximum = max(
        archetype_scores[p][archetype] for p in participants
    )

    for person in participants:

        if maximum > 0:
            score = archetype_scores[person][archetype] / maximum
        else:
            score = 0

        normalized_scores[person][archetype] = score


# Assign each archetype to at most one participant.
# Greedy assignment: highest normalized person-archetype pairs
# are selected first. Remaining participants receive their
# best remaining archetype. Ties follow the listed archetype order.

all_pairs = []

for person in participants:
    for archetype in archetypes:
        all_pairs.append((
            normalized_scores[person][archetype],
            person,
            archetype
        ))

all_pairs.sort(key=lambda item: item[0], reverse=True)

assigned_people = set()
assigned_archetypes = set()
final_archetypes = {}

for score, person, archetype in all_pairs:

    if person not in assigned_people and archetype not in assigned_archetypes:

        final_archetypes[person] = archetype
        assigned_people.add(person)
        assigned_archetypes.add(archetype)

        if len(assigned_people) == len(participants):
            break


print("\n" + "=" * 65)
print("                PERSONALITY ARCHETYPES")
print("=" * 65)

for person in participants:

    archetype = final_archetypes[person]

    if archetype == "THE SPAMMER":
        detail = f"Average burst: {calculate_burst(person):.2f}"

    elif archetype == "THE GROUP MOM":
        detail = f"Caring keywords: {caring_score(person)}"

    elif archetype == "THE NIGHT OWL":
        detail = f"Night messages: {night_owl_score(person)*100:.1f}%"

    elif archetype == "THE STORYTELLER":
        detail = f"Average words: {storyteller_score(person):.2f}"

    elif archetype == "THE DRAMA QUEEN":
        detail = f"Drama messages: {drama_score(person)*100:.1f}%"

    elif archetype == "THE GHOST":
        detail = f"Silent days: {silent_streaks[person]} longest consecutive days"

    elif archetype == "THE COMEDIAN":
        detail = f"Funny word ratio: {comedian_score(person)*100:.1f}%"

    else:
        detail = f"Question ratio: {question_score(person)*100:.1f}%"

    print(f"{person:<10} -> {archetype:<24} ({detail})")


# ------------------------------------------------------------
# FEATURE 8: FINAL GROUPDNA REPORT
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print('       GROUPDNA REPORT - "HOSTEL BOIS 4EVER"')
print("=" * 70)

print(f"{total_days} days | {total_messages:,} messages | "
      f"{len(participants)} members")

print("=" * 70)

print("\nGROUP STATISTICS")

print("Date range      :", first_date.strftime("%d %B %Y"),
      "to", last_date.strftime("%d %B %Y"))

print("Busiest day     :", busiest_day.strftime("%d %B %Y"))
print("Busiest hour    :", f"{busiest_hour:02d}:00")

print("\nMESSAGES PER PERSON")

for person in sorted_people:

    percentage = message_counts[person] / total_messages * 100

    bar = "█" * int(20 * message_counts[person] / max(message_counts.values()))

    print(f"{person:<10} {bar:<22} "
          f"{message_counts[person]:>5} ({percentage:.1f}%)")


print("\nACTIVITY HEATMAP")

print("Person    ", end="")

for hour in range(24):
    print(f"{hour:02d} ", end="")

print()

for i, person in enumerate(participants):

    maximum = np.max(heatmap[i])

    print(f"{person:<10}", end="")

    for hour in range(24):

        value = heatmap[i][hour]

        if maximum == 0 or value == 0:
            symbol = "."
        elif value <= maximum * 0.25:
            symbol = "░"
        elif value <= maximum * 0.50:
            symbol = "▒"
        else:
            symbol = "█"

        print(f"{symbol:>2} ", end="")

    print()


print("\nTOP 10 GROUP WORDS")

for word, count in top_words:

    bar = "█" * int(20 * count / top_words[0][1])

    print(f"{word:<15} {bar:<22} {count}")


print("\nRESPONSE PATTERNS")

if valid_responses:
    print("Fastest replier:", fastest)
    print("Slowest replier:", slowest)

for person in sorted(
    participants,
    key=lambda p: silent_streaks[p],
    reverse=True
):
    print(f"{person:<10}: {silent_streaks[person]} silent days")


print("\nPERSONALITY ARCHETYPES")

for person in participants:
    print(f"{person:<10} -> {final_archetypes[person]}")


print("\nDATA CLEANING SUMMARY")

print("System messages skipped:", system_count)
print("Deleted messages:", sum(deleted_count.values()))
print("Media messages:", sum(media_count.values()))

Please upload hostel_bois.txt


Saving hostel_bois.txt to hostel_bois (1).txt

PARSER RESULTS
Successfully parsed: 3159 messages
Participants: 6
System messages skipped: 4
Deleted messages: 15
Media messages: 32

FIRST 5 MESSAGES
01/04/24, 01:17 Rahul scene fix
01/04/24, 01:17 Rahul haan
01/04/24, 01:18 Rahul kya scene
01/04/24, 02:13 Rahul abhi free hai?
01/04/24, 02:13 Rahul abey

LAST 5 MESSAGES
30/05/24, 19:14 Priya Take care everyone
30/05/24, 19:28 Priya Karan that sounds tough, take care
30/05/24, 21:17 Aman the existential dread is back
30/05/24, 21:30 Karan Long day guys, woke up at six for that placement workshop which started at eight by the way classic, then ran around campus collecting signatures for the project document, ate lunch at four PM standing in the corridor, came back to hostel and realized I forgot my charger in the lab.
30/05/24, 23:31 Aman anyone awake?

                 GROUP OVERVIEW
Group              : Hostel Bois 4ever
Period             : 01 April 2024 to 30 May 2024
Total days        

# Project Reflection

The GroupDNA project helped me understand how Python fundamentals can be used to perform real-world data analysis.

**Hardest Part:**  
The most challenging part was parsing WhatsApp chat data and handling different message formats, followed by calculating response times and silent streaks.

**What I Would Do Differently:**  
I would improve the visualization, optimize the code and add more detailed analysis of group communication patterns.

**What I Learned:**  
- File handling and data parsing
- Lists and dictionaries
- Functions and loops
- NumPy arrays and indexing
- Word frequency analysis
- Date and time calculations
- Data visualization using text characters
- Conditional logic and data analysis

**My Personality Archetype:**  
To be filled in after running the project on my own WhatsApp chat.